Required files

Knowledge bases:
1. passport_clean_kb.json
2. nid_clean_kb.json
3. tin_clean_kb.json
4. birth_death_clean_kb.json

Test QA files:
1. passport_test.json
2. nid_test.json
3. tin_test.json
4. birth_death_test.json

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentence-transformers faiss-cpu rank-bm25 rapidfuzz pandas tqdm

import os, re, json, pickle, random, time
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import faiss
from rank_bm25 import BM25Okapi
from rapidfuzz import fuzz
from google.colab import drive

drive.mount("/content/drive")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

BASE_DIR = Path("/content")
ART_DIR = Path("/content/drive/MyDrive/govt_rag_artifacts")
ART_DIR.mkdir(parents=True, exist_ok=True)

KB_FILES = {
    "passport": BASE_DIR / "passport_clean_kb.json",
    "birth_death": BASE_DIR / "birth_death_clean_kb.json",
    "nid": BASE_DIR / "nid_clean_kb.json",
    "tin": BASE_DIR / "TIN_clean_kb.json",
}

TEST_FILES = {
    "passport": BASE_DIR / "passport_qa_test.json",
    "birth_death": BASE_DIR / "birth_death_qa_test.json",
    "nid": BASE_DIR / "nid_qa_test.json",
    "tin": BASE_DIR / "TIN_qa_test.json",
}

print("Artifact folder:", ART_DIR)
print("\nKB files:")
for k, v in KB_FILES.items():
    print(k, v.exists(), v)

print("\nTest files:")
for k, v in TEST_FILES.items():
    print(k, v.exists(), v)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 124.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 99.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.6/676.6 kB 50.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
das

In [ ]:

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def normalize_domain(domain):
    d = str(domain).lower()
    if "passport" in d:
        return "passport"
    if "birth" in d or "death" in d:
        return "birth_death"
    if "nid" in d:
        return "nid"
    if "tin" in d or "tax" in d:
        return "tin"
    return d

def word_chunks(text, max_words=280, overlap=45):
    words = str(text).split()
    if len(words) <= max_words:
        return [str(text).strip()]

    chunks = []
    start = 0
    while start < len(words):
        end = min(start + max_words, len(words))
        chunk = " ".join(words[start:end]).strip()
        if chunk:
            chunks.append(chunk)
        if end == len(words):
            break
        start = max(0, end - overlap)
    return chunks

def build_chunk_text(item, chunk_text):
    title = item.get("title", "")
    topic = item.get("topic", "")
    service = item.get("service", "")
    source = item.get("source_name", "")

    parts = []
    if title:
        parts.append(f"শিরোনাম: {title}")
    if topic:
        parts.append(f"টপিক: {topic}")
    if service:
        parts.append(f"সেবা: {service}")
    if source:
        parts.append(f"উৎস: {source}")
    parts.append(f"তথ্য: {chunk_text}")

    return "\n".join(parts)

all_chunks = []
chunk_id = 0

for domain_key, path in KB_FILES.items():
    data = load_json(path)

    for item in data:
        text = item.get("text") or item.get("content") or item.get("output") or ""
        text = str(text).strip()
        if not text:
            continue

        domain = normalize_domain(item.get("domain", domain_key))
        split_texts = word_chunks(text, max_words=280, overlap=45)

        for part_id, part_text in enumerate(split_texts):
            chunk_id += 1
            all_chunks.append({
                "chunk_id": f"chunk_{chunk_id:05d}",
                "doc_id": item.get("doc_id", ""),
                "domain": domain,
                "topic": item.get("topic", ""),
                "title": item.get("title", ""),
                "source_url": item.get("source_url", ""),
                "source_name": item.get("source_name", ""),
                "text": part_text,
                "chunk_text": build_chunk_text(item, part_text),
            })

chunks_path = ART_DIR / "merged_kb_chunks.json"
with open(chunks_path, "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=2)

pd.DataFrame(all_chunks).to_csv(ART_DIR / "merged_kb_chunks.csv", index=False)

print("Total chunks:", len(all_chunks))
print("Saved:", chunks_path)
pd.DataFrame(all_chunks).groupby("domain").size()

Total chunks: 516
Saved: /content/drive/MyDrive/govt_rag_artifacts/merged_kb_chunks.json


domain
birth_death     79
nid            325
passport        56
tin             56
dtype: int64

In [ ]:
from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = "BAAI/bge-m3"

chunks = load_json(ART_DIR / "merged_kb_chunks.json")
chunk_texts = [c["chunk_text"] for c in chunks]

print("Loading BGE-M3...")
embedder = SentenceTransformer(EMBED_MODEL_NAME, device="cuda" if torch.cuda.is_available() else "cpu")

print("Encoding chunks...")
embeddings = embedder.encode(
    chunk_texts,
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True
).astype("float32")

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

faiss.write_index(index, str(ART_DIR / "bge_m3_faiss.index"))
np.save(ART_DIR / "bge_m3_embeddings.npy", embeddings)

def tokenize_bn_en(text):
    text = str(text).lower()
    return re.findall(r"[\u0980-\u09FF]+|[a-zA-Z0-9]+", text)

bm25_corpus = [tokenize_bn_en(c["chunk_text"]) for c in chunks]
bm25 = BM25Okapi(bm25_corpus)

with open(ART_DIR / "bm25_index.pkl", "wb") as f:
    pickle.dump(bm25, f)

with open(ART_DIR / "rag_config.json", "w", encoding="utf-8") as f:
    json.dump({
        "embedding_model": EMBED_MODEL_NAME,
        "total_chunks": len(chunks),
        "faiss_index": "bge_m3_faiss.index",
        "bm25_index": "bm25_index.pkl",
        "chunk_file": "merged_kb_chunks.json"
    }, f, ensure_ascii=False, indent=2)

print("Saved FAISS, embeddings, BM25, config in:", ART_DIR)

Loading BGE-M3...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Encoding chunks...


Batches:   0%|          | 0/33 [00:00<?, ?it/s]

Saved FAISS, embeddings, BM25, config in: /content/drive/MyDrive/govt_rag_artifacts


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

chunks = load_json(ART_DIR / "merged_kb_chunks.json")
index = faiss.read_index(str(ART_DIR / "bge_m3_faiss.index"))

with open(ART_DIR / "bm25_index.pkl", "rb") as f:
    bm25 = pickle.load(f)

def hybrid_retrieve(question, domain=None, dense_k=80, bm25_k=80, final_k=5):
    domain = normalize_domain(domain) if domain else None

    # Dense retrieval
    q_emb = embedder.encode(
        [question],
        normalize_embeddings=True,
        show_progress_bar=False
    ).astype("float32")

    dense_scores, dense_ids = index.search(q_emb, dense_k)
    dense_ids = dense_ids[0].tolist()

    # BM25 retrieval
    q_tokens = tokenize_bn_en(question)
    bm25_scores = bm25.get_scores(q_tokens)
    bm25_ids = np.argsort(bm25_scores)[::-1][:bm25_k].tolist()

    scores = defaultdict(float)

    # RRF style score
    for rank, idx in enumerate(dense_ids):
        if idx < 0:
            continue
        scores[idx] += 1.0 / (60 + rank + 1)

    for rank, idx in enumerate(bm25_ids):
        scores[idx] += 1.0 / (60 + rank + 1)

    # Domain boost
    for idx in list(scores.keys()):
        if domain and chunks[idx]["domain"] == domain:
            scores[idx] += 0.025

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    if domain:
        domain_ranked = [(idx, sc) for idx, sc in ranked if chunks[idx]["domain"] == domain]
        if len(domain_ranked) >= final_k:
            ranked = domain_ranked

    selected = []
    for idx, score in ranked[:final_k]:
        item = dict(chunks[idx])
        item["retrieval_score"] = float(score)
        selected.append(item)

    return selected

def build_context(retrieved_chunks):
    blocks = []
    for i, c in enumerate(retrieved_chunks, 1):
        blocks.append(
            f"[Context {i}]\n"
            f"Domain: {c.get('domain','')}\n"
            f"Title: {c.get('title','')}\n"
            f"Topic: {c.get('topic','')}\n"
            f"Source: {c.get('source_url','')}\n"
            f"{c.get('chunk_text','')}"
        )
    return "\n\n".join(blocks)

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading Qwen...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model.eval()

def qwen_rag_answer(question, domain, top_k=5, max_new_tokens=180):
    retrieved = hybrid_retrieve(question, domain=domain, final_k=top_k)
    context = build_context(retrieved)

    system_prompt = (
        "তুমি বাংলাদেশ সরকারি সেবা বিষয়ে সহায়ক। "
        "শুধুমাত্র দেওয়া Context ব্যবহার করে উত্তর দাও। "
        "উত্তর অবশ্যই পরিষ্কার, সংক্ষিপ্ত এবং বাংলায় হবে। "
        "Context-এ উত্তর না থাকলে বলবে: 'প্রদত্ত তথ্যে এই প্রশ্নের উত্তর পাওয়া যায়নি।'"
    )

    user_prompt = f"""
Context:
{context}

Question:
{question}

Answer in Bangla:
""".strip()

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            top_p=1.0,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id
        )

    gen = out[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(gen, skip_special_tokens=True).strip()

    return answer, retrieved

Loading Qwen...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [ ]:
def load_tests():
    rows = []
    for domain, path in TEST_FILES.items():
        if not path.exists():
            print("Missing:", path)
            continue

        data = load_json(path)
        for item in data:
            rows.append({
                "domain": domain,
                "id": item.get("id", ""),
                "question": item.get("instruction", ""),
                "gold": item.get("output", ""),
                "source_url": item.get("source_url", "")
            })
    return rows

def exact_match(pred, gold):
    return int(str(pred).strip() == str(gold).strip())

def token_f1(pred, gold):
    pred_tokens = tokenize_bn_en(pred)
    gold_tokens = tokenize_bn_en(gold)

    if len(pred_tokens) == 0 and len(gold_tokens) == 0:
        return 1.0
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return 0.0

    common = defaultdict(int)
    for t in gold_tokens:
        common[t] += 1

    match = 0
    for t in pred_tokens:
        if common[t] > 0:
            match += 1
            common[t] -= 1

    if match == 0:
        return 0.0

    precision = match / len(pred_tokens)
    recall = match / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

def evaluate_rows(df):
    return {
        "Exact Match": df["exact_match"].mean(),
        "Fuzzy Match": df["fuzzy"].mean(),
        "Token F1": df["token_f1"].mean(),
    }

test_rows = load_tests()
print("Total test rows:", len(test_rows))

# For quick debugging set MAX_TEST = 10.
# For final result set MAX_TEST = None.
MAX_TEST = None

if MAX_TEST:
    test_rows = test_rows[:MAX_TEST]

pred_rows = []
start_time = time.time()

for row in tqdm(test_rows):
    pred, retrieved = qwen_rag_answer(
        question=row["question"],
        domain=row["domain"],
        top_k=5,
        max_new_tokens=180
    )

    top_sources = [r.get("source_url", "") for r in retrieved]
    top_titles = [r.get("title", "") for r in retrieved]

    fuzzy_score = fuzz.token_set_ratio(pred, row["gold"]) / 100.0
    f1_score = token_f1(pred, row["gold"])
    em = exact_match(pred, row["gold"])

    pred_rows.append({
        "domain": row["domain"],
        "id": row["id"],
        "question": row["question"],
        "gold": row["gold"],
        "prediction": pred,
        "exact_match": em,
        "fuzzy": fuzzy_score,
        "token_f1": f1_score,
        "retrieved_titles": " || ".join(top_titles),
        "retrieved_sources": " || ".join(top_sources),
    })

elapsed = time.time() - start_time
df_pred = pd.DataFrame(pred_rows)

pred_path = ART_DIR / "qwen_rag_predictions.csv"
json_path = ART_DIR / "qwen_rag_predictions.json"
metrics_path = ART_DIR / "qwen_rag_metrics.csv"

df_pred.to_csv(pred_path, index=False)
df_pred.to_json(json_path, force_ascii=False, orient="records", indent=2)

overall_metrics = evaluate_rows(df_pred)

domain_metrics = []
for domain, g in df_pred.groupby("domain"):
    m = evaluate_rows(g)
    m["domain"] = domain
    m["count"] = len(g)
    domain_metrics.append(m)

df_metrics = pd.DataFrame(domain_metrics)
df_overall = pd.DataFrame([{
    "domain": "overall",
    "count": len(df_pred),
    **overall_metrics
}])

final_metrics = pd.concat([df_overall, df_metrics], ignore_index=True)
final_metrics.to_csv(metrics_path, index=False)

print("\n========== Qwen + RAG Results ==========")
print(final_metrics)

print("\nSaved predictions:", pred_path)
print("Saved metrics:", metrics_path)
print("Elapsed minutes:", round(elapsed / 60, 2))

print("\nBaseline note:")
print("Previous Qwen no-RAG fuzzy match = 0.405524")
print("Compare this with Qwen + RAG Fuzzy Match above.")

Total test rows: 248


  0%|          | 0/248 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



========== Qwen + RAG Results ==========
        domain  count  Exact Match  Fuzzy Match  Token F1
0      overall    248     0.004032     0.556982  0.276523
1  birth_death     63     0.000000     0.605774  0.340276
2          nid     74     0.000000     0.510195  0.226644
3     passport     63     0.000000     0.604362  0.311451
4          tin     48     0.020833     0.502887  0.223898

Saved predictions: /content/drive/MyDrive/govt_rag_artifacts/qwen_rag_predictions.csv
Saved metrics: /content/drive/MyDrive/govt_rag_artifacts/qwen_rag_metrics.csv
Elapsed minutes: 60.09

Baseline note:
Previous Qwen no-RAG fuzzy match = 0.405524
Compare this with Qwen + RAG Fuzzy Match above.


In [ ]:
!pip install -q evaluate rouge-score bert-score sacrebleu nltk

import pandas as pd
import numpy as np
import re
import nltk
from rapidfuzz import fuzz
from nltk.translate.meteor_score import meteor_score
from bert_score import score as bert_score

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

pred_path = ART_DIR / "qwen_rag_predictions.csv"
df = pd.read_csv(pred_path)

df["prediction"] = df["prediction"].fillna("").astype(str)
df["gold"] = df["gold"].fillna("").astype(str)

def bn_tokenize(text):
    return re.findall(r"[\u0980-\u09FF]+|[a-zA-Z0-9]+", str(text).lower())

def exact_match(pred, gold):
    return int(str(pred).strip() == str(gold).strip())

def token_f1_score(pred, gold):
    pred_tokens = bn_tokenize(pred)
    gold_tokens = bn_tokenize(gold)

    if len(pred_tokens) == 0 and len(gold_tokens) == 0:
        return 1.0
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return 0.0

    gold_counts = {}
    for t in gold_tokens:
        gold_counts[t] = gold_counts.get(t, 0) + 1

    match = 0
    for t in pred_tokens:
        if gold_counts.get(t, 0) > 0:
            match += 1
            gold_counts[t] -= 1

    if match == 0:
        return 0.0

    precision = match / len(pred_tokens)
    recall = match / len(gold_tokens)

    return 2 * precision * recall / (precision + recall)

def ngram_counts(tokens, n):
    return {
        tuple(tokens[i:i+n]): tokens[i:i+n]
        for i in range(len(tokens)-n+1)
    }

def rouge_n(pred, gold, n=1):
    pred_tokens = bn_tokenize(pred)
    gold_tokens = bn_tokenize(gold)

    if len(pred_tokens) < n or len(gold_tokens) < n:
        return 0.0

    pred_ngrams = list(zip(*[pred_tokens[i:] for i in range(n)]))
    gold_ngrams = list(zip(*[gold_tokens[i:] for i in range(n)]))

    pred_counts = {}
    gold_counts = {}

    for g in pred_ngrams:
        pred_counts[g] = pred_counts.get(g, 0) + 1

    for g in gold_ngrams:
        gold_counts[g] = gold_counts.get(g, 0) + 1

    overlap = 0
    for g in pred_counts:
        overlap += min(pred_counts[g], gold_counts.get(g, 0))

    if overlap == 0:
        return 0.0

    precision = overlap / max(1, len(pred_ngrams))
    recall = overlap / max(1, len(gold_ngrams))

    return 2 * precision * recall / (precision + recall)

def lcs_len(x, y):
    m, n = len(x), len(y)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m):
        for j in range(n):
            if x[i] == y[j]:
                dp[i+1][j+1] = dp[i][j] + 1
            else:
                dp[i+1][j+1] = max(dp[i][j+1], dp[i+1][j])

    return dp[m][n]

def rouge_l(pred, gold):
    pred_tokens = bn_tokenize(pred)
    gold_tokens = bn_tokenize(gold)

    if not pred_tokens or not gold_tokens:
        return 0.0

    lcs = lcs_len(pred_tokens, gold_tokens)

    precision = lcs / len(pred_tokens)
    recall = lcs / len(gold_tokens)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)

def bleu_score_single(pred, gold):
    pred_tokens = bn_tokenize(pred)
    gold_tokens = bn_tokenize(gold)

    if not pred_tokens or not gold_tokens:
        return 0.0

    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
    smooth = SmoothingFunction().method1

    return sentence_bleu(
        [gold_tokens],
        pred_tokens,
        smoothing_function=smooth,
        weights=(0.25, 0.25, 0.25, 0.25)
    )

def meteor_single(pred, gold):
    pred_tokens = bn_tokenize(pred)
    gold_tokens = bn_tokenize(gold)

    if not pred_tokens or not gold_tokens:
        return 0.0

    try:
        return meteor_score([gold_tokens], pred_tokens)
    except:
        return 0.0

df["Exact Match"] = df.apply(lambda x: exact_match(x["prediction"], x["gold"]), axis=1)
df["Fuzzy Match"] = df.apply(lambda x: fuzz.token_set_ratio(x["prediction"], x["gold"]) / 100.0, axis=1)
df["Token F1"] = df.apply(lambda x: token_f1_score(x["prediction"], x["gold"]), axis=1)
df["BLEU"] = df.apply(lambda x: bleu_score_single(x["prediction"], x["gold"]), axis=1)
df["ROUGE1"] = df.apply(lambda x: rouge_n(x["prediction"], x["gold"], n=1), axis=1)
df["ROUGE2"] = df.apply(lambda x: rouge_n(x["prediction"], x["gold"], n=2), axis=1)
df["ROUGEL"] = df.apply(lambda x: rouge_l(x["prediction"], x["gold"]), axis=1)
df["METEOR"] = df.apply(lambda x: meteor_single(x["prediction"], x["gold"]), axis=1)

print("Computing BERTScore...")

P, R, F1 = bert_score(
    df["prediction"].tolist(),
    df["gold"].tolist(),
    model_type="bert-base-multilingual-cased",
    lang="bn",
    verbose=True,
    batch_size=16,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

df["BERT Precision"] = P.cpu().numpy()
df["BERT Recall"] = R.cpu().numpy()
df["BERT F1"] = F1.cpu().numpy()

metrics = {
    "Exact Match": df["Exact Match"].mean(),
    "Fuzzy Match": df["Fuzzy Match"].mean(),
    "Token F1": df["Token F1"].mean(),
    "BLEU": df["BLEU"].mean(),
    "ROUGE1": df["ROUGE1"].mean(),
    "ROUGE2": df["ROUGE2"].mean(),
    "ROUGEL": df["ROUGEL"].mean(),
    "METEOR": df["METEOR"].mean(),
    "BERT Precision": df["BERT Precision"].mean(),
    "BERT Recall": df["BERT Recall"].mean(),
    "BERT F1": df["BERT F1"].mean(),
}

rag_metrics_df = pd.DataFrame(
    [{"Metric": k, "Score": v} for k, v in metrics.items()]
)

display(rag_metrics_df)

rag_metrics_df.to_csv(ART_DIR / "qwen_rag_full_metrics.csv", index=False)
df.to_csv(ART_DIR / "qwen_rag_predictions_with_full_metrics.csv", index=False)

print("Saved full metrics:", ART_DIR / "qwen_rag_full_metrics.csv")
print("Saved full prediction metrics:", ART_DIR / "qwen_rag_predictions_with_full_metrics.csv")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.5 MB/s eta 0:00:00
Computing BERTScore...


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

...

computing greedy matching.


  0%|          | 0/16 [00:00<?, ?it/s]

done in 1.47 seconds, 168.97 sentences/sec


,Metric,Score
0,Exact Match,0.004032
1,Fuzzy Match,0.556982
2,Token F1,0.276523
3,BLEU,0.076780
4,ROUGE1,0.276523
5,ROUGE2,0.140934
6,ROUGEL,0.242742
7,METEOR,0.238961
8,BERT Precision,0.750183
9,BERT Recall,0.734679


Saved full metrics: /content/drive/MyDrive/govt_rag_artifacts/qwen_rag_full_metrics.csv
Saved full prediction metrics: /content/drive/MyDrive/govt_rag_artifacts/qwen_rag_predictions_with_full_metrics.csv


In [ ]:
baseline_qwen = {
    "Model": "Qwen",
    "Setting": "No RAG",
    "Exact Match": 0.000000,
    "Fuzzy Match": 0.405524,
    "BLEU": 0.039033,
    "ROUGE1": 0.045052,
    "ROUGE2": 0.005184,
    "ROUGEL": 0.044337,
    "METEOR": 0.164995,
    "BERT Precision": 0.705691,
    "BERT Recall": 0.713622,
    "BERT F1": 0.708823,
}

rag_qwen = {
    "Model": "Qwen",
    "Setting": "BGE-M3 + BM25 RAG",
}

for k, v in metrics.items():
    rag_qwen[k] = v

comparison_df = pd.DataFrame([baseline_qwen, rag_qwen])

display(comparison_df)

comparison_df.to_csv(ART_DIR / "qwen_baseline_vs_rag_comparison.csv", index=False)

print("Saved:", ART_DIR / "qwen_baseline_vs_rag_comparison.csv")

,Model,Setting,Exact Match,Fuzzy Match,BLEU,ROUGE1,ROUGE2,ROUGEL,METEOR,BERT Precision,BERT Recall,BERT F1,Token F1
0,Qwen,No RAG,0.000000,0.405524,0.039033,0.045052,0.005184,0.044337,0.164995,0.705691,0.713622,0.708823,NaN
1,Qwen,BGE-M3 + BM25 RAG,0.004032,0.556982,0.076780,0.276523,0.140934,0.242742,0.238961,0.750183,0.734679,0.741129,0.276523


Saved: /content/drive/MyDrive/govt_rag_artifacts/qwen_baseline_vs_rag_comparison.csv


In [1]:
# ============================================================
# INDEPENDENT RAG EVALUATION CELL
# No previous notebook cells need to be executed.
# ============================================================

%pip install -q bert-score==0.3.13 sacrebleu rapidfuzz nltk

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
from collections import Counter
import re
import unicodedata

import pandas as pd
import torch
import nltk

from rapidfuzz import fuzz
from nltk.translate.meteor_score import meteor_score
from bert_score import score as bert_score
from sacrebleu.metrics import BLEU
from IPython.display import display


# ------------------------------------------------------------
# 1. File locations
# ------------------------------------------------------------

ART_DIR = Path("/content/drive/MyDrive/govt_rag_artifacts")

# Existing raw prediction file
INPUT_FILE = ART_DIR / "qwen_rag_predictions.csv"

# New final files
OUTPUT_PREDICTIONS = ART_DIR / "RAG_predictions.csv"
OUTPUT_RESULTS = ART_DIR / "RAG_results.csv"


if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Could not find the input file:\n{INPUT_FILE}\n\n"
        "Check the Drive folder name and file name."
    )


# ------------------------------------------------------------
# 2. Load existing generated answers
# ------------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

required_columns = {
    "id",
    "domain",
    "question",
    "gold",
    "prediction"
}

missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(
        f"Required columns are missing: {sorted(missing_columns)}\n"
        f"Available columns: {df.columns.tolist()}"
    )

for column in required_columns:
    df[column] = df[column].fillna("").astype(str)

print(f"Loaded {len(df)} saved RAG predictions.")


# ------------------------------------------------------------
# 3. Bangla-English text normalization
# ------------------------------------------------------------

BANGLA_TO_ENGLISH_DIGITS = str.maketrans(
    "০১২৩৪৫৬৭৮৯",
    "0123456789"
)


def normalize_text(text):
    """
    Normalize Bangla/English text for lexical metrics.

    Operations:
    - Unicode normalization
    - Bangla digits converted to English digits
    - Lowercasing English letters
    - Punctuation removal
    - Extra whitespace removal
    """

    text = unicodedata.normalize("NFKC", str(text))
    text = text.translate(BANGLA_TO_ENGLISH_DIGITS)
    text = text.lower()

    # Retain Bangla, English letters and numbers
    text = re.sub(
        r"[^\u0980-\u09FFA-Za-z0-9]+",
        " ",
        text
    )

    return re.sub(r"\s+", " ", text).strip()


def tokenize(text):
    normalized = normalize_text(text)

    if not normalized:
        return []

    return normalized.split()


# ------------------------------------------------------------
# 4. Evaluation metric functions
# ------------------------------------------------------------

def normalized_exact_match(prediction, reference):
    return int(
        normalize_text(prediction) ==
        normalize_text(reference)
    )


def token_f1(prediction, reference):
    pred_tokens = tokenize(prediction)
    ref_tokens = tokenize(reference)

    if not pred_tokens and not ref_tokens:
        return 1.0

    if not pred_tokens or not ref_tokens:
        return 0.0

    pred_counter = Counter(pred_tokens)
    ref_counter = Counter(ref_tokens)

    common_tokens = sum(
        (pred_counter & ref_counter).values()
    )

    if common_tokens == 0:
        return 0.0

    precision = common_tokens / len(pred_tokens)
    recall = common_tokens / len(ref_tokens)

    return (
        2 * precision * recall /
        (precision + recall)
    )


def fuzzy_match(prediction, reference):
    return (
        fuzz.token_set_ratio(
            normalize_text(prediction),
            normalize_text(reference)
        ) / 100.0
    )


def rouge_n_f1(prediction, reference, n):
    pred_tokens = tokenize(prediction)
    ref_tokens = tokenize(reference)

    if len(pred_tokens) < n or len(ref_tokens) < n:
        return 0.0

    pred_ngrams = Counter(
        tuple(pred_tokens[i:i+n])
        for i in range(len(pred_tokens) - n + 1)
    )

    ref_ngrams = Counter(
        tuple(ref_tokens[i:i+n])
        for i in range(len(ref_tokens) - n + 1)
    )

    overlap = sum(
        (pred_ngrams & ref_ngrams).values()
    )

    if overlap == 0:
        return 0.0

    precision = overlap / sum(pred_ngrams.values())
    recall = overlap / sum(ref_ngrams.values())

    return (
        2 * precision * recall /
        (precision + recall)
    )


def lcs_length(sequence_1, sequence_2):
    """
    Memory-efficient longest common subsequence.
    """

    previous = [0] * (len(sequence_2) + 1)

    for token_1 in sequence_1:
        current = [0]

        for j, token_2 in enumerate(sequence_2, start=1):

            if token_1 == token_2:
                current.append(previous[j - 1] + 1)

            else:
                current.append(
                    max(previous[j], current[j - 1])
                )

        previous = current

    return previous[-1]


def rouge_l_f1(prediction, reference):
    pred_tokens = tokenize(prediction)
    ref_tokens = tokenize(reference)

    if not pred_tokens or not ref_tokens:
        return 0.0

    lcs = lcs_length(pred_tokens, ref_tokens)

    precision = lcs / len(pred_tokens)
    recall = lcs / len(ref_tokens)

    if precision + recall == 0:
        return 0.0

    return (
        2 * precision * recall /
        (precision + recall)
    )


# Download resources needed by NLTK METEOR
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)


def meteor_value(prediction, reference):
    pred_tokens = tokenize(prediction)
    ref_tokens = tokenize(reference)

    if not pred_tokens or not ref_tokens:
        return 0.0

    try:
        return float(
            meteor_score(
                [ref_tokens],
                pred_tokens
            )
        )

    except Exception as error:
        raise RuntimeError(
            f"METEOR calculation failed: {error}"
        ) from error


# ------------------------------------------------------------
# 5. Prepare final prediction table
# ------------------------------------------------------------

rag_predictions = pd.DataFrame({
    "id": df["id"],
    "domain": df["domain"],
    "question": df["question"],
    "gold_answer": df["gold"],
    "generated_answer": df["prediction"]
})


# ------------------------------------------------------------
# 6. Separate retrieved titles and sources into columns
# ------------------------------------------------------------

def split_retrieved_items(value, maximum_items=5):

    if pd.isna(value) or not str(value).strip():
        return [""] * maximum_items

    items = [
        item.strip()
        for item in str(value).split("||")
        if item.strip()
    ]

    items = items[:maximum_items]

    return items + [""] * (maximum_items - len(items))


for rank in range(1, 6):
    rag_predictions[f"retrieved_title_{rank}"] = ""
    rag_predictions[f"retrieved_source_{rank}"] = ""


if "retrieved_titles" in df.columns:

    retrieved_title_rows = df[
        "retrieved_titles"
    ].apply(split_retrieved_items)

    for index in range(5):

        rag_predictions[
            f"retrieved_title_{index + 1}"
        ] = retrieved_title_rows.apply(
            lambda items, i=index: items[i]
        )


if "retrieved_sources" in df.columns:

    retrieved_source_rows = df[
        "retrieved_sources"
    ].apply(split_retrieved_items)

    for index in range(5):

        rag_predictions[
            f"retrieved_source_{index + 1}"
        ] = retrieved_source_rows.apply(
            lambda items, i=index: items[i]
        )


# ------------------------------------------------------------
# 7. Calculate per-answer lexical metrics
# ------------------------------------------------------------

prediction_reference_pairs = zip(
    rag_predictions["generated_answer"],
    rag_predictions["gold_answer"]
)

pairs = list(prediction_reference_pairs)


rag_predictions["Normalized Exact Match"] = [
    normalized_exact_match(pred, gold)
    for pred, gold in pairs
]

rag_predictions["Token F1"] = [
    token_f1(pred, gold)
    for pred, gold in pairs
]

rag_predictions["Fuzzy Match"] = [
    fuzzy_match(pred, gold)
    for pred, gold in pairs
]

rag_predictions["ROUGE-1 F1"] = [
    rouge_n_f1(pred, gold, n=1)
    for pred, gold in pairs
]

rag_predictions["ROUGE-2 F1"] = [
    rouge_n_f1(pred, gold, n=2)
    for pred, gold in pairs
]

rag_predictions["ROUGE-L F1"] = [
    rouge_l_f1(pred, gold)
    for pred, gold in pairs
]

rag_predictions["METEOR"] = [
    meteor_value(pred, gold)
    for pred, gold in pairs
]


# ------------------------------------------------------------
# 8. Calculate BERTScore
# ------------------------------------------------------------

print("\nComputing BERTScore...")

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

bert_predictions = [
    unicodedata.normalize("NFKC", text).strip()
    for text in rag_predictions["generated_answer"]
]

bert_references = [
    unicodedata.normalize("NFKC", text).strip()
    for text in rag_predictions["gold_answer"]
]


bert_precision, bert_recall, bert_f1_scores = bert_score(
    bert_predictions,
    bert_references,

    # Keep this model unchanged for all four experiments
    model_type="bert-base-multilingual-cased",

    batch_size=16,
    device=device,
    verbose=True,

    idf=False,
    rescale_with_baseline=False
)


rag_predictions["BERT Precision"] = (
    bert_precision.cpu().numpy()
)

rag_predictions["BERT Recall"] = (
    bert_recall.cpu().numpy()
)

rag_predictions["BERT F1"] = (
    bert_f1_scores.cpu().numpy()
)


# ------------------------------------------------------------
# 9. Corpus BLEU
# ------------------------------------------------------------

# Corpus BLEU is calculated over the complete dataset.
# It is not calculated independently for every row.

bleu_metric = BLEU(
    tokenize="none",
    smooth_method="exp",
    effective_order=True
)


def calculate_corpus_bleu(group):

    predictions = [
        " ".join(tokenize(text))
        for text in group["generated_answer"]
    ]

    references = [
        " ".join(tokenize(text))
        for text in group["gold_answer"]
    ]

    if not any(predictions) or not any(references):
        return 0.0

    bleu_result = bleu_metric.corpus_score(
        predictions,
        [references]
    )

    # SacreBLEU produces a 0–100 score.
    # Divide by 100 to produce a 0–1 score.
    return bleu_result.score / 100.0


# ------------------------------------------------------------
# 10. Create overall and per-domain result table
# ------------------------------------------------------------

def create_result_rows(group, scope, domain):

    metric_scores = {
        "Normalized Exact Match":
            group["Normalized Exact Match"].mean(),

        "Token F1":
            group["Token F1"].mean(),

        "Fuzzy Match":
            group["Fuzzy Match"].mean(),

        "Corpus BLEU":
            calculate_corpus_bleu(group),

        "ROUGE-1 F1":
            group["ROUGE-1 F1"].mean(),

        "ROUGE-2 F1":
            group["ROUGE-2 F1"].mean(),

        "ROUGE-L F1":
            group["ROUGE-L F1"].mean(),

        "METEOR":
            group["METEOR"].mean(),

        "BERT Precision":
            group["BERT Precision"].mean(),

        "BERT Recall":
            group["BERT Recall"].mean(),

        "BERT F1":
            group["BERT F1"].mean()
    }

    return [
        {
            "scope": scope,
            "domain": domain,
            "count": len(group),
            "metric": metric,
            "score": float(score)
        }
        for metric, score in metric_scores.items()
    ]


result_rows = create_result_rows(
    rag_predictions,
    scope="overall",
    domain="all"
)


for domain, domain_group in rag_predictions.groupby(
    "domain",
    sort=True
):

    result_rows.extend(
        create_result_rows(
            domain_group,
            scope="domain",
            domain=domain
        )
    )


rag_results = pd.DataFrame(result_rows)


# ------------------------------------------------------------
# 11. Save final files
# ------------------------------------------------------------

# utf-8-sig helps Bangla display properly in Excel.
rag_predictions.to_csv(
    OUTPUT_PREDICTIONS,
    index=False,
    encoding="utf-8-sig"
)

rag_results.to_csv(
    OUTPUT_RESULTS,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 12. Display final results
# ------------------------------------------------------------

print("\nEvaluation complete.")
print(f"BERTScore device: {device}")
print(f"\nSaved predictions:\n{OUTPUT_PREDICTIONS}")
print(f"\nSaved results:\n{OUTPUT_RESULTS}")

print("\n========== Overall RAG Results ==========")

overall_results = rag_results[
    rag_results["scope"] == "overall"
][["metric", "score"]].reset_index(drop=True)

display(overall_results)

print("\n========== Prediction File Preview ==========")

display(rag_predictions.head(3))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 80.6 MB/s eta 0:00:00
Mounted at /content/drive
Loaded 248 saved RAG predictions.

Computing BERTScore...


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

...

  0%|          | 0/16 [00:00<?, ?it/s]

done in 2.24 seconds, 110.96 sentences/sec

Evaluation complete.
BERTScore device: cuda

Saved predictions:
/content/drive/MyDrive/govt_rag_artifacts/RAG_predictions.csv

Saved results:
/content/drive/MyDrive/govt_rag_artifacts/RAG_results.csv

========== Overall RAG Results ==========


,metric,score
0,Normalized Exact Match,0.008065
1,Token F1,0.294431
2,Fuzzy Match,0.590451
3,Corpus BLEU,0.117717
4,ROUGE-1 F1,0.294431
5,ROUGE-2 F1,0.158889
6,ROUGE-L F1,0.257748
7,METEOR,0.258441
8,BERT Precision,0.757424
9,BERT Recall,0.739752



========== Prediction File Preview ==========


,id,domain,question,gold_answer,generated_answer,retrieved_title_1,retrieved_source_1,retrieved_title_2,retrieved_source_2,retrieved_title_3,...,Normalized Exact Match,Token F1,Fuzzy Match,ROUGE-1 F1,ROUGE-2 F1,ROUGE-L F1,METEOR,BERT Precision,BERT Recall,BERT F1
0,passport_001,passport,ই-পাসপোর্ট আবেদনের পাঁচটি মূল ধাপ কী কী?,ই-পাসপোর্ট আবেদনের ৫টি সহজ ধাপ হলো: ১. বর্তমান...,ই-পাসপোর্ট আবেদনের পাঁচটি মূল ধাপ হলো: প্রথমে ...,ই-পাসপোর্ট আবেদনের পাঁচটি মূল ধাপ,https://www.epassport.gov.bd/instructions/five...,ই-পাসপোর্ট সেবার সারসংক্ষেপ,https://www.epassport.gov.bd/landing,চালুকৃত ই-পাসপোর্ট অফিস যাচাই,...,0,0.432432,0.693069,0.432432,0.166667,0.324324,0.249780,0.811092,0.765431,0.787600
1,passport_179,passport,প্রবাসী শ্রমিকের ৬৪ পৃষ্ঠা ৫ বছর এক্সপ্রেস ই-প...,বিদেশস্থ বাংলাদেশ মিশনে শ্রমিক ও শিক্ষার্থীদের...,প্রদত্ত তথ্যে এই প্রশ্নের উত্তর পাওয়া যায়নি।,বিদেশস্থ বাংলাদেশ মিশনের সাধারণ আবেদনকারীদের ই...,https://www.epassport.gov.bd/instructions/pass...,বাংলাদেশের অভ্যন্তরে ৬৪ পৃষ্ঠা ৫ বছর মেয়াদী ই...,https://www.epassport.gov.bd/instructions/pass...,বিদেশস্থ বাংলাদেশ মিশনে শ্রমিক ও শিক্ষার্থীদের...,...,0,0.000000,0.289017,0.000000,0.000000,0.000000,0.000000,0.624629,0.589136,0.606364
2,passport_216,passport,বিদেশে শ্রমিক শিক্ষার্থীর ৬৪ পৃষ্ঠা ৫ বছর এক্স...,বিদেশস্থ বাংলাদেশ মিশনে শ্রমিক ও শিক্ষার্থীদের...,বিদেশে শ্রমিক শিক্ষার্থীর ৬৪ পৃষ্ঠা ৫ বছর এক্স...,বিদেশস্থ বাংলাদেশ মিশনে শ্রমিক ও শিক্ষার্থীদের...,https://www.epassport.gov.bd/instructions/pass...,ই-পাসপোর্ট ফি নির্ধারণের ভিত্তি,https://www.epassport.gov.bd/instructions/pass...,বিদেশস্থ বাংলাদেশ মিশনের সাধারণ আবেদনকারীদের ই...,...,0,0.411765,0.666667,0.411765,0.125000,0.411765,0.283391,0.836097,0.762501,0.797605
